# Super Tic-Tac-Toe RL Benchmark Notebook

This Colab notebook is a guided experiment plan for the supplied `board.py`, `env.py`, `models.py`, `train.py`, and `evaluate.py` project files.

It does **not** train anything until you run the training cells yourself in Google Colab. The goal is to benchmark several agents under the same environment rules:

1. Random baseline
2. Heuristic baseline
3. Tabular Q-learning / minimax-style Q-learning baseline
4. DQN-style neural Q agent
5. PPO self-play agent using the project training script
6. Rollout MCTS baseline

For stochastic games, always report averages over many games and multiple seeds.

## Literature-grounded plan

The literature review motivates a progression from simple to stronger methods:

- **Classic search and minimax**: useful conceptually, but exhaustive search is too expensive for 96 cells and stochastic transitions.
- **Q-learning**: a simple model-free baseline. It learns from sampled games and can handle stochastic outcomes, but tabular state storage is too large for strong play.
- **DQN**: replaces the Q-table with a neural network and uses replay plus target networks.
- **Actor-critic / PPO**: learns a policy and value function directly; this is a good fit for legal action masks and self-play.
- **MCTS / AlphaZero-style methods**: use simulations and value/policy guidance to improve decisions at play time.

This notebook benchmarks these ideas incrementally. The final report should compare them using the same metrics: win rate, loss rate, draw rate, average reward, game length, illegal losses, and forfeits.

## Literature references to cite in the report

Use the provided literature review as the reference source for these claims:

- Shannon/Turing/minimax/alpha-beta: motivates why pure search is a baseline but not sufficient for this large stochastic game.
- Watkins Q-learning and Tic-Tac-Toe Q-learning work: motivates tabular Q-learning and epsilon-greedy exploration.
- Hu and Wellman Nash Q-learning: motivates multi-agent concerns and non-stationarity in self-play.
- Coquelin and Munos UCT/MCTS: motivates rollout search and exploration bonuses.
- AlphaGo/AlphaZero: motivates policy-value networks, self-play, and MCTS guidance.
- DeepStack/CFR discussion: useful contrast for imperfect-information games; Super Tic-Tac-Toe has stochasticity but not hidden information.
- Sparse reward section: motivates legal-action masks, self-play, reward shaping discussion, and curriculum/baseline comparisons.

## Research-quality audit before training

For an assignment demo, one PPO self-play run and a random-opponent evaluation are enough to show that the environment and model run. For a paper-quality experiment, that is **not enough**.

The minimum credible protocol for this stochastic two-player game is:

- Use fixed environment code and unit tests before training.
- Train every learning method with at least 3 random seeds; 5 seeds is better.
- Evaluate on held-out evaluation seeds that are never used during training.
- Alternate X/O sides during evaluation.
- Compare against an opponent suite, not only random play.
- Report confidence intervals or standard errors, not only one win rate.
- Save raw game-level results so tables can be reproduced.
- Track wall-clock time and approximate environment steps, not only episodes.
- Include ablations: no heuristic, no masking, smaller network, random-agent training, self-play training.
- Explicitly state limitations: stochastic outcomes, sparse rewards, finite compute, and self-play non-stationarity.

The supplied `train.py` PPO implementation is a good **baseline implementation**. It is not a final publication-grade AlphaZero-style system. To claim stronger research contribution, add at least periodic evaluation, multi-seed reporting, checkpoint selection by validation performance, and preferably GAE/opponent pools/MCTS-guided self-play.

## Standards borrowed from prior work

Use the following standards when writing the methodology:

- **DQN standard**: experience replay and a target network are expected, following the deep Q-learning line of work.
- **PPO standard**: clipped policy objective, multiple minibatch epochs over on-policy rollouts, value loss, entropy bonus, and ideally GAE.
- **AlphaGo/AlphaZero standard**: self-play should be evaluated against previous checkpoints or strong search baselines, not only against itself.
- **MCTS standard**: report simulations per move and exploration constant; separate search-time compute from learned model strength.
- **Stochastic-game standard**: report repeated trials with uncertainty intervals because one game can be won or lost because of redirection randomness.
- **Benchmarking standard**: use the same evaluation suite for every agent and preserve raw results.

For this project, the strongest defensible claim is: *we implement and compare a suite of increasingly capable baselines for a stochastic board game, and PPO self-play is the main neural RL baseline.* Avoid claiming solved or optimal play unless large-scale evaluations support it.

## Before running in Colab

Upload the whole `super_tictactoe_rl` folder to Colab or Google Drive. The notebook expects these files to be available:

- `board.py`
- `env.py`
- `models.py`
- `train.py`
- `evaluate.py`
- `utils.py`

Recommended Colab workflow:

1. Zip the local project folder.
2. Upload/unzip it to `/content/super_tictactoe_rl`, or put it in Google Drive.
3. Run the setup cells below.

If package installation fails because Colab changed Python/TensorFlow versions, keep the environment and baseline sections; use the local project `requirements.txt` as the source of compatible package pins.

In [ ]:
# Colab setup. Run once per runtime.
# This cell may restart the runtime once after reinstalling NumPy/Pandas.
# That restart is intentional: it prevents binary ABI errors such as
# "numpy.dtype size changed, may indicate binary incompatibility".
import os, sys, subprocess
from pathlib import Path
os.environ.setdefault('TF_USE_LEGACY_KERAS', '1')
os.environ.setdefault('GYM_DISABLE_WARNINGS', '1')

marker = Path('/content/.super_ttt_deps_installed_v3')
req = Path('/content/super_tictactoe_rl/requirements-colab.txt')
if not marker.exists():
    if not req.exists():
        raise FileNotFoundError('Missing /content/super_tictactoe_rl/requirements-colab.txt. Re-upload the updated project folder.')
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q',
        '--upgrade', '--force-reinstall', '--no-cache-dir',
        '-r', str(req),
    ])
    marker.touch()
    print('Dependencies installed. Restarting runtime once; after restart, rerun from this cell.')
    os.kill(os.getpid(), 9)
else:
    print('Colab dependencies already installed for this runtime.')

# Optional TF-Agents install. Use this only if your Colab runtime supports it.
# The custom PPO trainer can run without tf-agents installed.
# %pip install -q --no-deps tf-agents==0.19.0 tensorflow-probability==0.23.0 gin-config

In [ ]:
from pathlib import Path
import sys, os, random, pickle, math, copy, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

candidate_dirs = [
    Path.cwd(),
    Path('/content/super_tictactoe_rl'),
    Path('/content/drive/MyDrive/super_tictactoe_rl'),
]
PROJECT_DIR = next((p for p in candidate_dirs if (p / 'env.py').exists()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError('Could not find env.py. Upload/unzip super_tictactoe_rl to /content or Drive first.')
sys.path.insert(0, str(PROJECT_DIR))
print('Using project directory:', PROJECT_DIR)

from board import SuperTicTacToeBoard, all_playable_coords
from env import SuperTicTacToeEnv
from utils import random_legal_action

SEED = 7
rng = np.random.default_rng(SEED)
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
# Environment sanity check.
env = SuperTicTacToeEnv(seed=SEED)
obs, info = env.reset()
print('Observation shape:', obs.shape)
print('Action count:', env.action_space.n)
print('Initial legal actions:', int(info['action_mask'].sum()))
print(env.render())

assert obs.shape == (97,)
assert env.action_space.n == 96
assert int(info['action_mask'].sum()) == 96

## Agent interface and baselines

Every agent below implements:

```python
select_action(env) -> int
```

The environment already knows whose turn it is through `env.current_player`. Legal-action masks are used so agents do not intentionally pick occupied cells.

In [ ]:
class RandomAgent:
    name = 'random'
    def __init__(self, seed=0):
        self.rng = np.random.default_rng(seed)
    def select_action(self, env):
        return random_legal_action(env.legal_action_mask(), self.rng)


def immediate_win_action(env, player):
    for action in env.board.legal_actions():
        coord = env.board.action_to_coord(action)
        board_copy = env.board.copy()
        board_copy.place(coord, player)
        if board_copy.check_winner() == player:
            return action
    return None


def centerish_score(coord):
    _, _, r, c = coord
    # In a 4x4 board, the four central cells are strongest by this simple heuristic.
    return -((r - 1.5) ** 2 + (c - 1.5) ** 2)


class HeuristicAgent:
    name = 'heuristic'
    def __init__(self, seed=0):
        self.rng = np.random.default_rng(seed)
    def select_action(self, env):
        player = env.current_player
        # 1. Take immediate direct win if available.
        action = immediate_win_action(env, player)
        if action is not None:
            return action
        # 2. Block opponent immediate direct win if available.
        action = immediate_win_action(env, -player)
        if action is not None and env.legal_action_mask()[action]:
            return action
        # 3. Prefer central local cells, with small random tie breaking.
        legal = env.board.legal_actions()
        scored = []
        for action in legal:
            coord = env.board.action_to_coord(action)
            scored.append((centerish_score(coord) + 1e-3 * self.rng.random(), action))
        return max(scored)[1]


In [ ]:
def play_game(agent_x, agent_o, seed=None, max_steps=200):
    env = SuperTicTacToeEnv(seed=seed)
    obs, info = env.reset(seed=seed)
    done = False
    steps = 0
    forfeits = 0
    illegal = False
    last_info = {'winner': 0, 'reason': 'not_finished'}
    while not done and steps < max_steps:
        agent = agent_x if env.current_player == 1 else agent_o
        action = agent.select_action(env)
        obs, reward, terminated, truncated, last_info = env.step(action)
        done = bool(terminated or truncated)
        steps += 1
        forfeits += int(bool(last_info.get('forfeited', False)))
        illegal = illegal or last_info.get('reason') == 'illegal_action'
    return {
        'winner': int(last_info['winner']),
        'steps': steps,
        'forfeits': forfeits,
        'illegal': illegal,
    }


def evaluate_matchup(agent_a, agent_b, games=100, seed=0):
    rows = []
    for i in tqdm(range(games), desc=f'{agent_a.name} vs {agent_b.name}'):
        # Alternate sides to avoid X/O bias.
        if i % 2 == 0:
            result = play_game(agent_a, agent_b, seed=seed + i)
            a_player = 1
        else:
            result = play_game(agent_b, agent_a, seed=seed + i)
            a_player = -1
        winner = result['winner']
        rows.append({
            'agent_a': agent_a.name,
            'agent_b': agent_b.name,
            'a_result': 'win' if winner == a_player else ('draw' if winner == 0 else 'loss'),
            **result,
        })
    df = pd.DataFrame(rows)
    summary = df['a_result'].value_counts(normalize=True).rename('rate').to_frame()
    summary['count'] = df['a_result'].value_counts()
    return df, summary


In [ ]:
random_agent = RandomAgent(seed=1)
heuristic_agent = HeuristicAgent(seed=2)

# Start with 100 games, then increase to 500 or 1000 for final reporting.
baseline_games = 100
baseline_df, baseline_summary = evaluate_matchup(heuristic_agent, random_agent, games=baseline_games, seed=1000)
display(baseline_summary)
baseline_df[['steps', 'forfeits']].describe()

## Tabular Q-learning baseline

Tabular Q-learning is included mainly as a benchmark and discussion point. The full state space is enormous, so this method may not become strong. It is still useful for showing why deep RL is needed.

For a two-player zero-sum turn-taking game, the non-terminal target uses the negative value of the next state because the next state is from the opponent's perspective:

```text
target = reward                      if terminal
target = -gamma * max_a Q(next, a)    otherwise
```

This is a simple minimax-style adaptation of Q-learning.

In [ ]:
from collections import defaultdict

def state_key(env):
    return tuple(env.get_observation().astype(np.int8).tolist())


class QTableAgent:
    name = 'q_table'
    def __init__(self, q_table, epsilon=0.0, seed=0):
        self.q_table = q_table
        self.epsilon = epsilon
        self.rng = np.random.default_rng(seed)
    def select_action(self, env):
        mask = env.legal_action_mask()
        legal = np.flatnonzero(mask)
        if self.rng.random() < self.epsilon:
            return int(self.rng.choice(legal))
        q = self.q_table[state_key(env)].copy()
        q[~mask] = -1e9
        return int(np.argmax(q))


def train_q_learning(
    episodes=50000,
    alpha=0.05,
    gamma=0.99,
    eps_start=0.90,
    eps_end=0.05,
    seed=0,
):
    q_table = defaultdict(lambda: np.zeros(96, dtype=np.float32))
    rng = np.random.default_rng(seed)
    metrics = []
    env = SuperTicTacToeEnv(seed=seed)
    for ep in tqdm(range(episodes), desc='Q-learning'):
        eps = eps_end + (eps_start - eps_end) * max(0.0, 1.0 - ep / episodes)
        obs, _ = env.reset(seed=int(rng.integers(0, 2**31 - 1)))
        done = False
        steps = 0
        forfeits = 0
        last_info = {'winner': 0}
        while not done:
            key = state_key(env)
            mask = env.legal_action_mask()
            legal = np.flatnonzero(mask)
            if rng.random() < eps:
                action = int(rng.choice(legal))
            else:
                q = q_table[key].copy()
                q[~mask] = -1e9
                action = int(np.argmax(q))
            obs, reward, terminated, truncated, last_info = env.step(action)
            done = bool(terminated or truncated)
            forfeits += int(bool(last_info.get('forfeited', False)))
            if done:
                target = reward
            else:
                next_key = state_key(env)
                next_mask = env.legal_action_mask()
                next_q = q_table[next_key].copy()
                next_q[~next_mask] = -1e9
                target = -gamma * float(np.max(next_q))
            q_table[key][action] += alpha * (target - q_table[key][action])
            steps += 1
        if (ep + 1) % max(1, episodes // 100) == 0:
            metrics.append({'episode': ep + 1, 'winner': int(last_info['winner']), 'steps': steps, 'forfeits': forfeits, 'epsilon': eps})
    return q_table, pd.DataFrame(metrics)


# Example run. Increase episodes for actual experiments.
# q_table, q_metrics = train_q_learning(episodes=50000, seed=SEED)
# q_metrics.plot(x='episode', y=['steps', 'forfeits'], subplots=True, figsize=(10, 5));
# with open(PROJECT_DIR / 'models' / 'q_table.pkl', 'wb') as f:
#     pickle.dump(dict(q_table), f)


## DQN-style neural Q agent

DQN replaces the Q-table with a neural network. Use this when tabular Q-learning becomes too memory-hungry. The same zero-sum target is used:

```text
target = reward                       if terminal
target = -gamma * max_a Q_target(next, a) otherwise
```

This cell defines the core components. Start with small episode counts, then scale up.

In [ ]:
import tensorflow as tf

class DQN(tf.keras.Model):
    def __init__(self, hidden=256):
        super().__init__()
        self.net = tf.keras.Sequential([
            tf.keras.layers.Dense(hidden, activation='relu'),
            tf.keras.layers.Dense(hidden, activation='relu'),
            tf.keras.layers.Dense(96),
        ])
    def call(self, x):
        return self.net(tf.cast(x, tf.float32))


class ReplayBuffer:
    def __init__(self, capacity=100000, seed=0):
        self.capacity = capacity
        self.data = []
        self.index = 0
        self.rng = np.random.default_rng(seed)
    def add(self, item):
        if len(self.data) < self.capacity:
            self.data.append(item)
        else:
            self.data[self.index] = item
        self.index = (self.index + 1) % self.capacity
    def sample(self, batch_size):
        idx = self.rng.choice(len(self.data), size=batch_size, replace=False)
        return [self.data[i] for i in idx]
    def __len__(self):
        return len(self.data)


def masked_argmax(q_values, mask):
    q = np.array(q_values, copy=True)
    q[~mask] = -1e9
    return int(np.argmax(q))


class DQNAgent:
    name = 'dqn'
    def __init__(self, model, epsilon=0.0, seed=0):
        self.model = model
        self.epsilon = epsilon
        self.rng = np.random.default_rng(seed)
    def select_action(self, env):
        mask = env.legal_action_mask()
        legal = np.flatnonzero(mask)
        if self.rng.random() < self.epsilon:
            return int(self.rng.choice(legal))
        q = self.model(env.get_observation()[None, :]).numpy()[0]
        return masked_argmax(q, mask)


def train_dqn(episodes=50000, gamma=0.99, lr=3e-4, batch_size=256, seed=0):
    rng = np.random.default_rng(seed)
    online = DQN()
    target = DQN()
    online(tf.zeros((1, 97)))
    target(tf.zeros((1, 97)))
    target.set_weights(online.get_weights())
    opt = tf.keras.optimizers.Adam(lr)
    replay = ReplayBuffer(seed=seed)
    env = SuperTicTacToeEnv(seed=seed)
    metrics = []
    for ep in tqdm(range(episodes), desc='DQN'):
        eps = max(0.05, 1.0 - ep / (0.7 * episodes))
        obs, _ = env.reset(seed=int(rng.integers(0, 2**31 - 1)))
        done = False
        steps = 0
        last_info = {'winner': 0}
        while not done:
            mask = env.legal_action_mask()
            if rng.random() < eps:
                action = random_legal_action(mask, rng)
            else:
                q = online(obs[None, :]).numpy()[0]
                action = masked_argmax(q, mask)
            next_obs, reward, terminated, truncated, last_info = env.step(action)
            done = bool(terminated or truncated)
            next_mask = env.legal_action_mask()
            replay.add((obs, action, reward, next_obs, done, mask, next_mask))
            obs = next_obs
            steps += 1
            if len(replay) >= batch_size:
                batch = replay.sample(batch_size)
                b_obs, b_action, b_reward, b_next, b_done, _, b_next_mask = map(np.array, zip(*batch))
                next_q = target(b_next).numpy()
                next_q[~b_next_mask.astype(bool)] = -1e9
                y = b_reward.astype(np.float32) + (1.0 - b_done.astype(np.float32)) * (-gamma * np.max(next_q, axis=1))
                with tf.GradientTape() as tape:
                    q_all = online(b_obs)
                    q_act = tf.gather(q_all, b_action.astype(np.int32), batch_dims=1)
                    loss = tf.reduce_mean(tf.square(tf.convert_to_tensor(y) - q_act))
                opt.apply_gradients(zip(tape.gradient(loss, online.trainable_variables), online.trainable_variables))
        if (ep + 1) % 250 == 0:
            target.set_weights(online.get_weights())
        if (ep + 1) % max(1, episodes // 100) == 0:
            metrics.append({'episode': ep + 1, 'winner': int(last_info['winner']), 'steps': steps, 'epsilon': eps})
    return online, pd.DataFrame(metrics)


# Example run. Increase episodes for actual experiments.
# dqn_model, dqn_metrics = train_dqn(episodes=50000, seed=SEED)
# dqn_model.save_weights(str(PROJECT_DIR / 'models' / 'dqn.weights.h5'))


## PPO self-play agent

The project already contains a PPO-style self-play trainer in `train.py`. This is the strongest fully implemented training path in the supplied code because it learns a policy and value function directly and supports legal-action masking.

Run a small command first, then scale the episode count for final experiments.

In [ ]:
# Smoke run in Colab. This should finish quickly.
# !python {PROJECT_DIR / 'train.py'} --episodes 20 --batch-episodes 4 --device auto --save-path {PROJECT_DIR / 'models' / 'ppo_smoke.pt'}

# Longer run for actual reporting. Adjust upward if you have more Colab time.
# !python {PROJECT_DIR / 'train.py'} --episodes 100000 --batch-episodes 32 --update-epochs 4 --minibatch-size 512 --lr 3e-4 --device auto --save-path {PROJECT_DIR / 'models' / 'super_ttt_agent.pt'}

# Evaluate trained PPO model.
# !python {PROJECT_DIR / 'evaluate.py'} --games 500 --model-path {PROJECT_DIR / 'models' / 'super_ttt_agent.pt'} --deterministic --device auto


## Rollout MCTS baseline

This is a lightweight Monte Carlo action-evaluation baseline. For each legal move, it runs random rollouts after that move and chooses the action with the best average outcome.

It is not full AlphaZero, but it gives a search-based baseline related to MCTS/UCT literature. Increase `rollouts_per_action` for stronger but slower play.

In [ ]:
def clone_env(src_env, seed=None):
    cloned = SuperTicTacToeEnv(seed=seed)
    cloned.board = src_env.board.copy()
    cloned.current_player = src_env.current_player
    cloned.rng = np.random.default_rng(seed)
    return cloned


def rollout_random(env, root_player, rng, max_steps=200):
    done = False
    steps = 0
    info = {'winner': 0}
    while not done and steps < max_steps:
        action = random_legal_action(env.legal_action_mask(), rng)
        _, _, terminated, truncated, info = env.step(action)
        done = bool(terminated or truncated)
        steps += 1
    winner = int(info['winner'])
    if winner == root_player:
        return 1.0
    if winner == 0:
        return 0.0
    return -1.0


class RolloutMCTSAgent:
    def __init__(self, rollouts_per_action=8, seed=0):
        self.name = f'rollout_mcts_{rollouts_per_action}'
        self.rollouts_per_action = rollouts_per_action
        self.rng = np.random.default_rng(seed)
    def select_action(self, env):
        root_player = env.current_player
        legal = env.board.legal_actions()
        scores = []
        for action in legal:
            total = 0.0
            for _ in range(self.rollouts_per_action):
                sim = clone_env(env, seed=int(self.rng.integers(0, 2**31 - 1)))
                _, _, terminated, truncated, info = sim.step(action)
                if terminated or truncated:
                    winner = int(info['winner'])
                    total += 1.0 if winner == root_player else (-1.0 if winner != 0 else 0.0)
                else:
                    total += rollout_random(sim, root_player, self.rng)
            scores.append((total / self.rollouts_per_action, action))
        return max(scores)[1]


# Example benchmark. Increase games cautiously because MCTS is slower.
# mcts_agent = RolloutMCTSAgent(rollouts_per_action=8, seed=3)
# mcts_df, mcts_summary = evaluate_matchup(mcts_agent, heuristic_agent, games=50, seed=3000)
# display(mcts_summary)


## Benchmarking protocol

Use the same evaluation function for all agents. For each matchup:

- Alternate X/O sides.
- Use at least 100 games for development and 500+ games for final reporting.
- Repeat with multiple random seeds.
- Report mean and standard deviation for stochastic results.

Suggested final table columns:

- Agent
- Opponent
- Games
- Win rate
- Loss rate
- Draw rate
- Average game length
- Average forfeits
- Illegal losses

In [ ]:
def summarize_df(df):
    return pd.Series({
        'games': len(df),
        'win_rate': (df['a_result'] == 'win').mean(),
        'loss_rate': (df['a_result'] == 'loss').mean(),
        'draw_rate': (df['a_result'] == 'draw').mean(),
        'avg_steps': df['steps'].mean(),
        'avg_forfeits': df['forfeits'].mean(),
        'illegal_rate': df['illegal'].mean(),
    })


# Example comparison table after training agents:
# rows = []
# for agent in [heuristic_agent, QTableAgent(q_table), DQNAgent(dqn_model), RolloutMCTSAgent(8)]:
#     df, _ = evaluate_matchup(agent, random_agent, games=200, seed=9000)
#     rows.append({'agent': agent.name, **summarize_df(df).to_dict()})
# results = pd.DataFrame(rows).sort_values('win_rate', ascending=False)
# display(results)
# results.plot(x='agent', y=['win_rate', 'loss_rate', 'draw_rate'], kind='bar', figsize=(10, 4));


## Multi-seed evaluation and confidence intervals

A publishable stochastic-game result should not be a single win-rate number. The helper below runs several independent evaluation seeds and reports a 95% normal-approximation interval for win/loss/draw rates. For the final report, use at least 500 games per seed when compute permits.

In [ ]:
def rate_ci(values, z=1.96):
    values = np.asarray(values, dtype=np.float64)
    mean = values.mean()
    if len(values) <= 1:
        return mean, np.nan, np.nan
    se = values.std(ddof=1) / np.sqrt(len(values))
    return mean, mean - z * se, mean + z * se


def multi_seed_matchup(agent_factory_a, agent_factory_b, games_per_seed=200, seeds=(0, 1, 2, 3, 4)):
    all_rows = []
    per_seed = []
    for seed in seeds:
        agent_a = agent_factory_a(seed)
        agent_b = agent_factory_b(seed + 10_000)
        df, _ = evaluate_matchup(agent_a, agent_b, games=games_per_seed, seed=seed * 100_000)
        df['eval_seed'] = seed
        all_rows.append(df)
        summary = summarize_df(df)
        summary['eval_seed'] = seed
        per_seed.append(summary)
    raw = pd.concat(all_rows, ignore_index=True)
    seed_df = pd.DataFrame(per_seed)
    rows = []
    for metric in ['win_rate', 'loss_rate', 'draw_rate', 'avg_steps', 'avg_forfeits', 'illegal_rate']:
        mean, lo, hi = rate_ci(seed_df[metric].values)
        rows.append({'metric': metric, 'mean': mean, 'ci95_low': lo, 'ci95_high': hi})
    return raw, seed_df, pd.DataFrame(rows)


# Example final-style baseline evaluation:
# raw, by_seed, ci = multi_seed_matchup(
#     lambda s: HeuristicAgent(seed=s),
#     lambda s: RandomAgent(seed=s),
#     games_per_seed=500,
#     seeds=(0, 1, 2, 3, 4),
# )
# display(ci)
# raw.to_csv(PROJECT_DIR / 'models' / 'heuristic_vs_random_raw.csv', index=False)


## Opponent suite for final benchmarking

Do not evaluate only against random play. Use a suite:

1. Random agent
2. Heuristic agent
3. Tabular Q-learning checkpoint, if trained
4. DQN checkpoint, if trained
5. PPO checkpoint
6. Rollout MCTS with fixed simulations per move

For each trained agent, report performance as both X and O. Also include cross-play: PPO versus DQN, PPO versus heuristic, MCTS versus PPO, etc. This follows the spirit of game-AI benchmarking where a method must beat multiple baselines, not just a weak random policy.

## Recommended final training budget

Use small runs only for debugging. For final Colab experiments, use a budget table like this:

| Method | Debug run | Final run | Notes |
|---|---:|---:|---|
| Heuristic | no training | no training | deterministic baseline plus stochastic environment |
| Q-table | 5k episodes | 50k-100k episodes | likely weak due to huge state space |
| DQN | 5k episodes | 100k-300k episodes | replay + target network; evaluate often |
| PPO | 1k episodes | 100k-500k episodes | main neural self-play baseline |
| Rollout MCTS | no training | 8-64 rollouts/action | slow; report compute budget |

If Colab time is limited, prioritize: heuristic baseline, random baseline, PPO self-play, and rollout MCTS. DQN and Q-table are useful for comparison and discussion but may require more tuning.

## Limitations to state honestly

- The PPO trainer in `train.py` uses Monte Carlo outcome returns rather than full GAE; this is acceptable as a baseline but should be described accurately.
- The self-play opponent is the current policy, not a league or checkpoint pool. That can cause non-stationarity and forgetting.
- The heuristic agent only checks direct immediate wins/blocks and does not model stochastic redirection probabilities.
- The DQN implementation here is intentionally compact for the assignment; production DQN would add prioritized replay, double DQN, gradient clipping, and more monitoring.
- The rollout MCTS baseline samples stochastic transitions but does not implement a full UCT tree with transpositions.
- Because the game is stochastic, evaluation variance can be large; confidence intervals are mandatory for strong claims.

These limitations do not invalidate the project. They make the claims precise: this is a benchmark study of several practical agents under a custom stochastic board-game environment.

## Report template

Use this structure in the final write-up:

1. **Environment**: Describe the 96-cell pyramid board, stochastic move resolution, legal-action masks, and sparse terminal rewards.
2. **Baselines**: Random and heuristic agents establish reference performance.
3. **Tabular Q-learning**: Explain Bellman updates, epsilon-greedy exploration, and why the state space limits performance.
4. **DQN**: Explain function approximation, replay, target networks, and how this addresses the large state space.
5. **PPO / actor-critic**: Explain policy-value learning, self-play, and why direct policy optimization fits legal action masks.
6. **MCTS**: Explain rollout search and how it relates to UCT and AlphaGo/AlphaZero-style methods.
7. **Results**: Include win/loss/draw tables, average game length, forfeits, illegal losses, and learning curves.
8. **Discussion**: Discuss stochasticity, sparse rewards, exploration, and which method performed best.

Important note: because move outcomes are stochastic, single games are not meaningful evidence. Report averages over repeated games and seeds.